In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

PSAHSE 1 PART B: includes 7 Simple Models + 3 Experiments (PCA, NMF, SMOTE).

In [1]:
!pip install --upgrade scikit-learn imbalanced-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 74.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 240.0/240.0 kB 13.9 MB/s eta 0:00:00
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.2.2
    Uninstalling scikit-learn-1.2.2:
      Successfully uninstalled scikit-learn-1.2.2
  Attempting uninstall: imbalanced-learn
    Found existing installation: imbalanced-learn 0.13.0
    Uninstalling imbalanced-learn-0.13.0:
      Successfully uninstalled imbalanced-learn-0.13.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
category-encoders 2.7.0 requires scikit-learn<1.6.0,>=1.0.0, but you have scikit-learn 1.8.0 which is incompatible.
cesium 0.12.4 requires numpy<3.0,>=2.0, but you have numpy 1.26.4 which is incompatible.
sklearn-compat 0.1.3 requires scikit-learn<1.7,>=1.2, but you have scikit-learn 

# HYPER-PARAMETER TUNING USING OPTUNA

In [3]:
# 
#  HYPERPARAMETER TUNING ( 9 MODELS) USES TPE SAMPLER AND MEDIAN PRUNER
# 

import optuna
import shap
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier
from sklearn.linear_model import RidgeClassifier, SGDClassifier, LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier # <--- NEW ADDITION
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner

# Settings
optuna.logging.set_verbosity(optuna.logging.WARNING)
RANDOM_STATE = 42

# 1. SETUP DATA
if 'X_corr_dropped' not in locals():
    print("WARNING: Using X_train (Fallback). Run Phase 1A first!")
    X_tune = X_train
    y_tune = y_labels
else:
    X_tune = X_corr_dropped
    y_tune = y_labels

print(f"Tuning on Data Shape: {X_tune.shape}")

# 2. SETUP OPTUNA
sampler = TPESampler(seed=RANDOM_STATE, n_startup_trials=5)
pruner = MedianPruner(n_startup_trials=5, n_warmup_steps=0, interval_steps=1)

def run_pruned_cv(trial, pipeline, X, y):
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
    fold_scores = []
    
    for step, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
        
        pipeline.fit(X_tr, y_tr)
        preds = pipeline.predict(X_val)
        score = balanced_accuracy_score(y_val, preds)
        fold_scores.append(score)
        
        trial.report(np.mean(fold_scores), step)
        if trial.should_prune():
            raise optuna.TrialPruned()
            
    return np.mean(fold_scores)

# =============================================================================
# PART A: TUNING SIMPLE MODELS
# =============================================================================
print("\n=== PART A: TUNING SIMPLE MODELS ===")

# 1. LOGISTIC REGRESSION
def obj_logreg(trial):
    C = trial.suggest_float('C', 1e-4, 1e2, log=True)
    solver = trial.suggest_categorical('solver', ['liblinear', 'lbfgs'])
    model = LogisticRegression(C=C, solver=solver, class_weight='balanced', max_iter=1000, random_state=RANDOM_STATE)
    pipeline = Pipeline([('scaler', StandardScaler()), ('model', model)])
    return run_pruned_cv(trial, pipeline, X_tune, y_tune)

study_logreg = optuna.create_study(direction='maximize', sampler=sampler, pruner=pruner)
study_logreg.optimize(obj_logreg, n_trials=15)
print(f"Logistic Regression Best: {study_logreg.best_value:.4f}")

# 2. RIDGE CLASSIFIER
def obj_ridge(trial):
    alpha = trial.suggest_float('alpha', 0.01, 100.0, log=True)
    pipeline = Pipeline([('scaler', StandardScaler()), ('model', RidgeClassifier(class_weight='balanced', alpha=alpha))])
    return run_pruned_cv(trial, pipeline, X_tune, y_tune)

study_ridge = optuna.create_study(direction='maximize', sampler=sampler, pruner=pruner)
study_ridge.optimize(obj_ridge, n_trials=15)
print(f"Ridge Best: {study_ridge.best_value:.4f}")

# 3. LINEAR SVM
def obj_svm(trial):
    params = {
        'alpha': trial.suggest_float('alpha', 1e-6, 1e-1, log=True),
        'penalty': trial.suggest_categorical('penalty', ['l2', 'l1', 'elasticnet']),
        'loss': 'hinge', 'class_weight': 'balanced', 'max_iter': 1000, 'random_state': RANDOM_STATE
    }
    pipeline = Pipeline([('scaler', StandardScaler()), ('model', SGDClassifier(**params))])
    return run_pruned_cv(trial, pipeline, X_tune, y_tune)

study_svm = optuna.create_study(direction='maximize', sampler=sampler, pruner=pruner)
study_svm.optimize(obj_svm, n_trials=15)
print(f"SVM Best: {study_svm.best_value:.4f}")

# 4. KNN
def obj_knn(trial):
    n_neighbors = trial.suggest_int('n_neighbors', 3, 20)
    weights = trial.suggest_categorical('weights', ['uniform', 'distance'])
    pipeline = Pipeline([('scaler', StandardScaler()), ('model', KNeighborsClassifier(n_neighbors=n_neighbors, weights=weights))])
    return run_pruned_cv(trial, pipeline, X_tune, y_tune)

study_knn = optuna.create_study(direction='maximize', sampler=sampler, pruner=pruner)
study_knn.optimize(obj_knn, n_trials=10)
print(f"KNN Best: {study_knn.best_value:.4f}")

# 5. GAUSSIAN NB
def obj_gnb(trial):
    var_smoothing = trial.suggest_float('var_smoothing', 1e-11, 1e-5, log=True)
    pipeline = Pipeline([('scaler', StandardScaler()), ('model', GaussianNB(var_smoothing=var_smoothing))])
    return run_pruned_cv(trial, pipeline, X_tune, y_tune)

study_gnb = optuna.create_study(direction='maximize', sampler=sampler, pruner=pruner)
study_gnb.optimize(obj_gnb, n_trials=10)
print(f"GaussianNB Best: {study_gnb.best_value:.4f}")

# =============================================================================
# PART B: TUNING COMPLEX MODELS (ENSEMBLES)
# =============================================================================
print("\n=== PART B: TUNING ENSEMBLES ===")

# 6. RANDOM FOREST (Bagging) - NEW!
print("   -> Tuning Random Forest...")
def obj_rf(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'max_depth': trial.suggest_int('max_depth', 5, 30),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
        'class_weight': 'balanced',
        'random_state': RANDOM_STATE,
        'n_jobs': -1
    }
    # Note: RF is parallel, so n_jobs=-1 in model, but we need to manage CPU usage in CV
    pipeline = Pipeline([('scaler', StandardScaler()), ('model', RandomForestClassifier(**params))])
    return run_pruned_cv(trial, pipeline, X_tune, y_tune)

study_rf = optuna.create_study(direction='maximize', sampler=sampler, pruner=pruner)
study_rf.optimize(obj_rf, n_trials=15)
print(f"Random Forest Best: {study_rf.best_value:.4f}")

# 7. LIGHTGBM
def obj_lgbm(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 200, 600),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.15),
        'num_leaves': trial.suggest_int('num_leaves', 20, 100),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
        'class_weight': 'balanced', 'random_state': RANDOM_STATE, 'verbose': -1,
        'device': 'gpu', 'gpu_platform_id': 0, 'gpu_device_id': 0
    }
    pipeline = Pipeline([('scaler', StandardScaler()), ('model', lgb.LGBMClassifier(**params))])
    return run_pruned_cv(trial, pipeline, X_tune, y_tune)

study_lgbm = optuna.create_study(direction='maximize', sampler=sampler, pruner=pruner)
study_lgbm.optimize(obj_lgbm, n_trials=20)
print(f"LightGBM Best: {study_lgbm.best_value:.4f}")

# 8. CATBOOST
def obj_cat(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 300, 1000),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2),
        'depth': trial.suggest_int('depth', 4, 10),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1, 10, log=True),
        'random_strength': trial.suggest_float('random_strength', 1, 20),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0, 1),
        'border_count': trial.suggest_int('border_count', 32, 255),
        'auto_class_weights': 'Balanced', 'verbose': 0, 'allow_writing_files': False, 'random_seed': RANDOM_STATE,
        'task_type': 'GPU', 'devices': '0'
    }
    model = CatBoostClassifier(**params)
    pipeline = Pipeline([('scaler', StandardScaler()), ('model', model)])
    return run_pruned_cv(trial, pipeline, X_tune, y_tune)

study_cat = optuna.create_study(direction='maximize', sampler=sampler, pruner=pruner)
study_cat.optimize(obj_cat, n_trials=15)
print(f"CatBoost Best: {study_cat.best_value:.4f}")

# 9. XGBOOST
print("   -> Tuning XGBoost...")
def obj_xgb(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 200, 600),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'gamma': trial.suggest_float('gamma', 0.1, 1.0),
        'scale_pos_weight': trial.suggest_float('scale_pos_weight', 40, 60),
        'random_state': RANDOM_STATE,
        'device': 'cuda', 'tree_method': 'hist' 
    }
    pipeline = Pipeline([('scaler', StandardScaler()), ('model', xgb.XGBClassifier(**params))])
    return run_pruned_cv(trial, pipeline, X_tune, y_tune)

study_xgb = optuna.create_study(direction='maximize', sampler=sampler, pruner=pruner)
study_xgb.optimize(obj_xgb, n_trials=15)
print(f"XGBoost Best: {study_xgb.best_value:.4f}")

# =============================================================================
# FINAL REPORT TABLE
# =============================================================================
results_data = {
    'Model': ['Logistic Regression', 'Ridge', 'Linear SVM', 'KNN', 'GaussianNB', 'Random Forest', 'LightGBM', 'CatBoost', 'XGBoost'],
    'Best Tuned Score': [
        study_logreg.best_value, study_ridge.best_value, study_svm.best_value, study_knn.best_value, 
        study_gnb.best_value, study_rf.best_value, study_lgbm.best_value, study_cat.best_value, study_xgb.best_value
    ],
    'Best Parameters': [
        str(study_logreg.best_params), str(study_ridge.best_params), str(study_svm.best_params), 
        str(study_knn.best_params), str(study_gnb.best_params), str(study_rf.best_params),
        str(study_lgbm.best_params), str(study_cat.best_params), str(study_xgb.best_params)
    ]
}

final_df = pd.DataFrame(results_data).sort_values(by='Best Tuned Score', ascending=False)
print("\n=== FINAL HYPERPARAMETER TUNING RESULTS ===")
print(final_df[['Model', 'Best Tuned Score']])
final_df.to_csv('final_tuning_report.csv', index=False)

Tuning on Data Shape: (60000, 142)

=== PART A: TUNING SIMPLE MODELS ===
Logistic Regression Best: 0.9554
Ridge Best: 0.9563


Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.


SVM Best: 0.9429
KNN Best: 0.7971
GaussianNB Best: 0.9158

=== PART B: TUNING ENSEMBLES ===
   -> Tuning Random Forest...
Random Forest Best: 0.9520


1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.


LightGBM Best: 0.9563
CatBoost Best: 0.9607
   -> Tuning XGBoost...


[02:12:07] WARNING: /workspace/src/common/error_msg.cc:58: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.



XGBoost Best: 0.9572

=== FINAL HYPERPARAMETER TUNING RESULTS ===
                 Model  Best Tuned Score
7             CatBoost          0.960721
8              XGBoost          0.957199
6             LightGBM          0.956314
1                Ridge          0.956276
0  Logistic Regression          0.955429
5        Random Forest          0.952028
2           Linear SVM          0.942867
4           GaussianNB          0.915811
3                  KNN          0.797078


In [5]:
# =============================================================================
# GENERATE OPTUNA PLOTS (ALL MODELS)
# =============================================================================
import matplotlib.pyplot as plt
from optuna.visualization.matplotlib import plot_optimization_history, plot_param_importances


def save_optuna_plots(study, model_name):
    print(f"Generating Plots for {model_name}...")
    
    # 1. Optimization History
    try:
        plt.figure(figsize=(10, 6))
        plot_optimization_history(study)
        plt.title(f"{model_name} Optimization History")
        plt.tight_layout()
        plt.savefig(f'optuna_{model_name.lower()}_history.png')
        plt.close() # Close to free memory
        
        # 2. Hyperparameter Importance
        # (Only works if more than 1 param was tuned)
        if len(study.best_params) > 1:
            plt.figure(figsize=(10, 6))
            plot_param_importances(study)
            plt.title(f"{model_name} Hyperparameter Importance")
            plt.tight_layout()
            plt.savefig(f'optuna_{model_name.lower()}_importance.png')
            plt.close()
        else:
            print(f"   (Skipping Importance Plot for {model_name}: Only 1 parameter tuned)")
            
    except Exception as e:
        print(f"   Could not plot for {model_name}: {e}")

# GENERATE PLOTS
# (Assumes study objects are still in memory from Phase 3)
#if 'study_ridge' in locals(): save_optuna_plots(study_ridge, "Ridge")
#if 'study_logreg' in locals(): save_optuna_plots(study_logreg, "LogisticRegression")
#if 'study_svm' in locals(): save_optuna_plots(study_svm, "SVM")
#if 'study_cat' in locals(): save_optuna_plots(study_cat, "CatBoost")
if 'study_lgbm' in locals(): save_optuna_plots(study_lgbm, "LightGBM")

print("\nSUCCESS: All requested plots saved to output directory.")

Generating Plots for LightGBM...


optuna.visualization.matplotlib._optimization_history.plot_optimization_history is experimental (supported from v2.2.0). The interface can change in the future.
optuna.visualization.matplotlib._param_importances.plot_param_importances is experimental (supported from v2.2.0). The interface can change in the future.



SUCCESS: All requested plots saved to output directory.


<Figure size 1000x600 with 0 Axes>

<Figure size 1000x600 with 0 Axes>